## Carry Propagation Analysis

In [1]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0, 1')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
class AnalysisConfig:
    def __init__(self):
        self.__dict__.update(
            n_general_eval=2048, n_pairs=1024, min_digits=1, max_digits=7,
        )

    def to_dict(self):
        return dict(vars(self))

cfg = AnalysisConfig()

In [3]:
vocab = list('0123456789+=;_')
pad_token = '_'
pad_id = vocab.index(pad_token)
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = dict(enumerate(vocab))

def encode(text):
    return [stoi[ch] for ch in text]

def sample_number(num_digits, rng):
    if num_digits == 1:
        return rng.randint(0, 9)
    return rng.randint(10 ** (num_digits - 1), 10 ** num_digits - 1)

def make_example(rng, min_digits, max_digits):
    a = sample_number(rng.randint(min_digits, max_digits), rng)
    b = sample_number(rng.randint(min_digits, max_digits), rng)
    return f'{a}+{b}={a+b};', a, b

def has_carry(a, b):
    carry = 0
    while a > 0 or b > 0:
        total = a % 10 + b % 10 + carry
        if total >= 10:
            return True
        carry = total // 10
        a //= 10
        b //= 10
    return False

In [4]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, mlp_mult, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_mult * d_model), nn.GELU(),
            nn.Linear(mlp_mult * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask):
        h = self.ln_1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        return x + h + self.mlp(self.ln_2(x + h))

class GPTConfig:
    def __init__(self, vocab_size=None, max_seq_len=64, d_model=256, n_heads=8, n_layers=6, mlp_mult=4, dropout=0.0):
        self.__dict__.update(
            vocab_size=len(vocab) if vocab_size is None else vocab_size,
            max_seq_len=max_seq_len, d_model=d_model, n_heads=n_heads,
            n_layers=n_layers, mlp_mult=mlp_mult, dropout=dropout,
        )

    def to_dict(self):
        return dict(vars(self))

class LLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg.d_model, cfg.n_heads, cfg.mlp_mult, cfg.dropout) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def hidden(self, idx, keep_cache=False):
        bsz, seq_len = idx.shape
        pos = torch.arange(seq_len, device=idx.device).unsqueeze(0).expand(bsz, seq_len)
        mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=idx.device), diagonal=1)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        cache = []
        for block in self.blocks:
            x = block(x, mask)
            if keep_cache:
                cache.append(x.detach().clone())
        logits = self.head(self.ln_f(x))
        return (logits, cache) if keep_cache else logits

    def forward(self, idx):
        return self.hidden(idx)

    def forward_with_cache(self, idx):
        return self.hidden(idx, keep_cache=True)

In [5]:
def first_digit_token_id(value):
    return stoi[str(value)[0]]

def prompt_ids(prompt, device):
    return torch.tensor([encode(prompt)], dtype=torch.long, device=device)

def generate_counterfactual_pair(rng, min_digits, max_digits):
    while True:
        a = sample_number(rng.randint(min_digits, max_digits), rng)
        b = sample_number(rng.randint(min_digits, max_digits), rng)

        if a > b:
            a, b = b, a

        clean_sum = a + b
        max_len = max(len(str(a)), len(str(b)))

        if len(str(clean_sum)) <= max_len:
            continue

        target_low = 10 ** (max_len - 1)
        target_high = 10 ** max_len - 1
        min_b = 0 if len(str(b)) == 1 else 10 ** (len(str(b)) - 1)

        delta_min = clean_sum - target_high
        delta_max = min(clean_sum - target_low, b - min_b)

        if delta_min > delta_max:
            continue

        delta = rng.randint(delta_min, delta_max)
        b_corrupt = b - delta
        corrupt_sum = a + b_corrupt

        return {
            'a': a,
            'b_clean': b,
            'b_corrupt': b_corrupt,
            'clean_sum': clean_sum,
            'corrupt_sum': corrupt_sum,
        }



@torch.no_grad()
def first_token_prediction(model, prompt, device):
    return int(torch.argmax(model(prompt_ids(prompt, device))[0, -1]).item())

@torch.no_grad()
def build_usable_pairs(model, device, n_pairs, seed):
    rng = random.Random(seed)
    pairs = []
    incorrent_n = 0
    for _ in range(500000):
        if len(pairs) >= n_pairs:
            break
        row = generate_counterfactual_pair(rng, cfg.min_digits, cfg.max_digits)
        clean_prompt = f"{row['a']}+{row['b_clean']}="
        corrupt_prompt = f"{row['a']}+{row['b_corrupt']}="
        target = first_digit_token_id(row['clean_sum'])
        if first_token_prediction(model, clean_prompt, device) == target and first_token_prediction(model, corrupt_prompt, device) != target:
            row.update(target_token_id=target, clean_prompt=clean_prompt, corrupt_prompt=corrupt_prompt)
            pairs.append(row)
        else:
            incorrent_n += 1
    print("incorrent_n:", incorrent_n)
    return pairs

@torch.no_grad()
def run_with_single_patch(model, idx, layer_idx, pos_idx, replacement):
    def hook(_module, _inputs, output):
        patched = output.clone()
        patched[:, pos_idx, :] = replacement
        return patched
    handle = model.blocks[layer_idx].register_forward_hook(hook)
    result = model(idx)
    handle.remove()
    return result


@torch.no_grad()
def evaluate_generalization(model, device, n_samples, min_digits, max_digits, seed):
    rng = random.Random(seed)
    stats = {'all': [0, 0], 'carry': [0, 0], 'no_carry': [0, 0]}

    for _ in range(n_samples):
        _, a, b = make_example(rng, min_digits, max_digits)
        ok = int(first_token_prediction(model, f'{a}+{b}=', device) == first_digit_token_id(a + b))
        group = 'carry' if has_carry(a, b) else 'no_carry'
        length = max(len(str(a)), len(str(b)))
        for name in ['all', group]:
            stats[name][0] += ok
            stats[name][1] += 1

    out = {
        'first_token_acc_all': stats['all'][0] / max(stats['all'][1], 1),
        'first_token_acc_carry': stats['carry'][0] / max(stats['carry'][1], 1),
        'first_token_acc_no_carry': stats['no_carry'][0] / max(stats['no_carry'][1], 1),
        'n_total': float(stats['all'][1]), 'n_carry': float(stats['carry'][1]), 'n_no_carry': float(stats['no_carry'][1]),
    }
    return out

In [6]:
torch.manual_seed(1558)
random.seed(1558)
device = 'cuda'

payload = torch.load("small_llm_training_artifacts/llm.pt", map_location='cpu')
model_cfg = GPTConfig(**payload['config'])
model = LLM(model_cfg).to(device)
model.load_state_dict(payload['model_state'])
model.eval()

LLM(
  (tok_emb): Embedding(14, 256)
  (pos_emb): Embedding(50, 256)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (ln_1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ln_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=14, bias=False)
)

In [7]:
general_metrics = evaluate_generalization(model, device, cfg.n_general_eval, cfg.min_digits, cfg.max_digits, 1558)
pairs = build_usable_pairs(model, device, cfg.n_pairs, 1558 * 2)
print(len(pairs))

incorrent_n: 12
1024


In [8]:
print(json.dumps(general_metrics, indent=4))

{
    "first_token_acc_all": 0.9970703125,
    "first_token_acc_carry": 0.996116504854369,
    "first_token_acc_no_carry": 1.0,
    "n_total": 2048.0,
    "n_carry": 1545.0,
    "n_no_carry": 503.0
}


In [9]:
n_layers = model_cfg.n_layers
noising_scores = np.zeros((n_layers, 1), dtype=np.float64)
denoising_scores = np.zeros((n_layers, 1), dtype=np.float64)

for row in pairs:
    clean_ids = prompt_ids(row['clean_prompt'], device)
    corrupt_ids = prompt_ids(row['corrupt_prompt'], device)
    target = int(row['target_token_id'])
    clean_equals_pos = clean_ids.shape[1] - 1
    corrupt_equals_pos = corrupt_ids.shape[1] - 1
    logits_clean, cache_clean = model.forward_with_cache(clean_ids)
    logits_corrupt, cache_corrupt = model.forward_with_cache(corrupt_ids)
    clean_base = float(logits_clean[0, -1, target].item())
    corrupt_base = float(logits_corrupt[0, -1, target].item())

    for layer_idx in range(n_layers):
        noised = run_with_single_patch(model, clean_ids, layer_idx, clean_equals_pos, cache_corrupt[layer_idx][:, corrupt_equals_pos, :])
        denoised = run_with_single_patch(model, corrupt_ids, layer_idx, corrupt_equals_pos, cache_clean[layer_idx][:, clean_equals_pos, :])
        noising_scores[layer_idx, 0] += clean_base - float(noised[0, -1, target].item())
        denoising_scores[layer_idx, 0] += float(denoised[0, -1, target].item()) - corrupt_base

noising_scores /= len(pairs)
denoising_scores /= len(pairs)
best_layer = int(np.argmax(denoising_scores[:, 0]))

In [10]:
print(best_layer)

5


In [11]:
print("noising_scores:\n", noising_scores)
print("denoising_scores:\n", denoising_scores)

noising_scores:
 [[ 0.02396489]
 [ 0.04296903]
 [ 9.43367015]
 [10.74519026]
 [10.77559186]
 [10.78941195]]
denoising_scores:
 [[-3.01472168e-02]
 [ 7.96277593e-03]
 [ 9.20130041e+00]
 [ 1.07625446e+01]
 [ 1.07853603e+01]
 [ 1.07894119e+01]]


In [12]:
checks = {name: 0 for name in ['clean', 'corrupt', 'denoised', 'noised']}
for row in pairs:
    clean_ids = prompt_ids(row['clean_prompt'], device)
    corrupt_ids = prompt_ids(row['corrupt_prompt'], device)
    target = int(row['target_token_id'])
    _, cache_clean = model.forward_with_cache(clean_ids)
    _, cache_corrupt = model.forward_with_cache(corrupt_ids)
    preds = {
        'clean': model(clean_ids),
        'corrupt': model(corrupt_ids),
        'denoised': run_with_single_patch(model, corrupt_ids, best_layer, corrupt_ids.shape[1] - 1, cache_clean[best_layer][:, clean_ids.shape[1] - 1, :]),
        'noised': run_with_single_patch(model, clean_ids, best_layer, clean_ids.shape[1] - 1, cache_corrupt[best_layer][:, corrupt_ids.shape[1] - 1, :]),
    }
    for name, logits in preds.items():
        checks[name] += int(torch.argmax(logits[0, -1]).item() == target)

In [13]:
summary_metrics = {f'{name}_acc': value / len(pairs) for name, value in checks.items()}
print(json.dumps(summary_metrics, indent=4))

{
    "clean_acc": 1.0,
    "corrupt_acc": 0.0,
    "denoised_acc": 1.0,
    "noised_acc": 0.0
}


### Carry-propagation-specific ablation check

In [15]:
def carry_chain_len(a, b):
    carry = 0
    current = 0
    longest = 0
    while a > 0 or b > 0 or carry > 0:
        new_carry = int(a % 10 + b % 10 + carry >= 10)
        current = current + 1 if new_carry else 0
        longest = max(longest, current)
        carry = new_carry
        a //= 10
        b //= 10
    return longest

def carry_group(a, b):
    chain = carry_chain_len(a, b)
    result = "no_carry"
    if chain == 0:
        if str(a + b)[0] == "1":
            result = "starts_with_1_" + result
        return result
    elif chain == 1:
        result = 'single_carry'
    else:
        result = "propagation"
    if len(str(a + b)) == max(len(str(a)), len(str(b))):
        result = "inside_" + result
        if str(a + b)[0] == "1":
            result = "starts_with_1_" + result
    else:
        result = "outside_" + result
    return result


@torch.no_grad()
def build_carry_direction(model, pairs, device, layer_idx):
    diffs = []
    for row in pairs:
        clean_ids = prompt_ids(row['clean_prompt'], device)
        corrupt_ids = prompt_ids(row['corrupt_prompt'], device)
        clean_equals_pos = clean_ids.shape[1] - 1
        corrupt_equals_pos = corrupt_ids.shape[1] - 1
        _, clean_cache = model.forward_with_cache(clean_ids)
        _, corrupt_cache = model.forward_with_cache(corrupt_ids)
        diffs.append(clean_cache[layer_idx][:, clean_equals_pos, :] - corrupt_cache[layer_idx][:, corrupt_equals_pos, :])
    direction = torch.stack(diffs).mean(dim=0)
    return direction / (direction.norm() + 1e-8)

@torch.no_grad()
def run_without_direction(model, idx, layer_idx, pos_idx, direction):
    def hook(_module, _inputs, output):
        patched = output.clone()
        h = patched[:, pos_idx, :]
        patched[:, pos_idx, :] = h - (h * direction).sum(dim=-1, keepdim=True) * direction
        return patched
    handle = model.blocks[layer_idx].register_forward_hook(hook)
    try:
        return model(idx)
    finally:
        handle.remove()

@torch.no_grad()
def evaluate_carry_specificity(model, device, layer_idx, direction, n_samples, seed, min_digits, max_digits):
    rng = random.Random(seed)
    stats = {name: [0, 0, 0] for name in ['no_carry', 'outside_single_carry', 'outside_propagation', 'inside_single_carry', 'inside_propagation', 'starts_with_1_inside_single_carry', 'starts_with_1_inside_propagation', "starts_with_1_no_carry"]}

    for _ in tqdm(range(n_samples)):
        a = sample_number(rng.randint(min_digits, max_digits), rng)
        b = sample_number(rng.randint(min_digits, max_digits), rng)
        idx = prompt_ids(f'{a}+{b}=', device)
        target = first_digit_token_id(a + b)
        group = carry_group(a, b)
        base_pred = int(torch.argmax(model(idx)[0, -1]).item())
        ablated = run_without_direction(model, idx, layer_idx, idx.shape[1] - 1, direction)
        ablated_pred = int(torch.argmax(ablated[0, -1]).item())
        stats[group][0] += 1
        stats[group][1] += int(base_pred == target)
        stats[group][2] += int(ablated_pred == target)

    out = {}
    for group, (total, base_correct, ablated_correct) in stats.items():
        base_acc = base_correct / total
        ablated_acc = ablated_correct / total
        out[group] = {'n': float(total), 'base_acc': base_acc, 'ablated_acc': ablated_acc, 'drop': base_acc - ablated_acc}
    return out

carry_direction = build_carry_direction(model, pairs, device, best_layer)
specificity_metrics = evaluate_carry_specificity(model, device, best_layer, carry_direction, 10000, 123, 1, 6)

100%|██████████| 10000/10000 [01:09<00:00, 144.51it/s]


In [16]:
print(pd.DataFrame(specificity_metrics).T)

                                        n  base_acc  ablated_acc      drop
no_carry                           2584.0  0.997291     0.996904  0.000387
outside_single_carry                402.0  1.000000     0.221393  0.778607
outside_propagation                 706.0  1.000000     0.164306  0.835694
inside_single_carry                3244.0  0.996917     0.996609  0.000308
inside_propagation                 2265.0  0.998675     0.998234  0.000442
starts_with_1_inside_single_carry   327.0  1.000000     0.681957  0.318043
starts_with_1_inside_propagation    202.0  1.000000     0.722772  0.277228
starts_with_1_no_carry              270.0  1.000000     0.525926  0.474074
